In [2]:
import json

def load_json(path):
    with open(path, 'r', encoding='gbk') as f:
        return json.load(f)

def save_json(data, path):
    with open(path, 'w', encoding='gbk') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def merge_annotations(file1_path, file2_path, output_path):
    # 1. 加载数据
    data1 = load_json(file1_path)
    data2 = load_json(file2_path)

    # 2. 合并类别
    old_cat_to_new = {}          # key: (file_index, old_category_id)
    new_categories = []
    new_cat_id = 1               # 新类别 ID 从 1 开始

    # 处理文件1的类别
    for cat in data1['categories']:
        old_cat_to_new[(1, cat['id'])] = new_cat_id
        new_cat = cat.copy()
        new_cat['id'] = new_cat_id
        new_categories.append(new_cat)
        new_cat_id += 1

    # 处理文件2的类别
    for cat in data2['categories']:
        old_cat_to_new[(2, cat['id'])] = new_cat_id
        new_cat = cat.copy()
        new_cat['id'] = new_cat_id
        new_categories.append(new_cat)
        new_cat_id += 1

    # 3. 合并图片
    old_img_to_new = {}          # key: (file_index, old_image_id)
    new_images = []
    new_img_id = 1

    for img in data1['images']:
        old_img_to_new[(1, img['id'])] = new_img_id
        new_img = img.copy()
        new_img['id'] = new_img_id
        new_images.append(new_img)
        new_img_id += 1

    for img in data2['images']:
        old_img_to_new[(2, img['id'])] = new_img_id
        new_img = img.copy()
        new_img['id'] = new_img_id
        new_images.append(new_img)
        new_img_id += 1

    # 4. 合并标注
    new_annotations = []
    new_ann_id = 1

    for ann in data1['annotations']:
        new_ann = ann.copy()
        # 更新 image_id
        old_img_id = ann['image_id']
        new_ann['image_id'] = old_img_to_new[(1, old_img_id)]
        # 更新 category_id
        old_cat_id = ann['category_id']
        new_ann['category_id'] = old_cat_to_new[(1, old_cat_id)]
        # 更新标注自身 id
        new_ann['id'] = new_ann_id
        new_annotations.append(new_ann)
        new_ann_id += 1

    for ann in data2['annotations']:
        new_ann = ann.copy()
        old_img_id = ann['image_id']
        new_ann['image_id'] = old_img_to_new[(2, old_img_id)]
        old_cat_id = ann['category_id']
        new_ann['category_id'] = old_cat_to_new[(2, old_cat_id)]
        new_ann['id'] = new_ann_id
        new_annotations.append(new_ann)
        new_ann_id += 1

    # 5. 构建最终合并数据
    merged = {
        "images": new_images,
        "annotations": new_annotations,
        "categories": new_categories
    }

    save_json(merged, output_path)
    print(f"合并完成！新文件已保存至：{output_path}")

# # ========== 使用示例 ==========
# if __name__ == "__main__":
#     # 请修改为你的实际文件路径
#     file1 = r"D:\CK\Lua\训练数据\航空碳纤维辅贴\1.json"
#     file2 = r"D:\CK\Lua\训练数据\航空碳纤维辅贴\2.json"
#     output = r"D:\CK\Lua\训练数据\航空碳纤维辅贴\12.json"

#     merge_annotations(file1, file2, output)

In [6]:
def merge_annotations_by_name(file1_path, file2_path, output_path):
    data1 = load_json(file1_path)
    data2 = load_json(file2_path)

    # 2. 按名称合并类别（去重）
    name_to_new_id = {}
    new_categories = []
    new_cat_id = 1

    # 先处理文件1的类别
    for cat in data1['categories']:
        name = cat['name']
        if name not in name_to_new_id:      # 未出现过
            name_to_new_id[name] = new_cat_id
            new_cat = cat.copy()
            new_cat['id'] = new_cat_id
            new_categories.append(new_cat)
            new_cat_id += 1

    # 再处理文件2的类别
    for cat in data2['categories']:
        name = cat['name']
        if name not in name_to_new_id:      # 同名的不会再添加
            name_to_new_id[name] = new_cat_id
            new_cat = cat.copy()
            new_cat['id'] = new_cat_id
            new_categories.append(new_cat)
            new_cat_id += 1

    # 为了更新标注的 category_id，需要根据 (file_index, old_category_id) 找到 name
    # 先建立 old_cat_id -> name 的映射
    def build_id_to_name(categories):
        return {cat['id']: cat['name'] for cat in categories}

    id2name_1 = build_id_to_name(data1['categories'])
    id2name_2 = build_id_to_name(data2['categories'])

    # 3. 合并图片（不变）
    old_img_to_new = {}
    new_images = []
    new_img_id = 1

    for img in data1['images']:
        old_img_to_new[(1, img['id'])] = new_img_id
        new_img = img.copy()
        new_img['id'] = new_img_id
        new_images.append(new_img)
        new_img_id += 1

    for img in data2['images']:
        old_img_to_new[(2, img['id'])] = new_img_id
        new_img = img.copy()
        new_img['id'] = new_img_id
        new_images.append(new_img)
        new_img_id += 1

    # 4. 合并标注
    new_annotations = []
    new_ann_id = 1

    # 文件1的标注
    for ann in data1['annotations']:
        new_ann = ann.copy()
        # 更新 image_id
        new_ann['image_id'] = old_img_to_new[(1, ann['image_id'])]
        # 通过旧 category_id 找到类别名，再映射到新 category_id
        old_cat_id = ann['category_id']
        name = id2name_1[old_cat_id]
        new_ann['category_id'] = name_to_new_id[name]
        new_ann['id'] = new_ann_id
        new_annotations.append(new_ann)
        new_ann_id += 1

    # 文件2的标注
    for ann in data2['annotations']:
        new_ann = ann.copy()
        new_ann['image_id'] = old_img_to_new[(2, ann['image_id'])]
        old_cat_id = ann['category_id']
        name = id2name_2[old_cat_id]
        new_ann['category_id'] = name_to_new_id[name]
        new_ann['id'] = new_ann_id
        new_annotations.append(new_ann)
        new_ann_id += 1

    merged = {
        "images": new_images,
        "annotations": new_annotations,
        "categories": new_categories
    }
    save_json(merged, output_path)
    print(f"合并完成（按名称去重）！新文件已保存至：{output_path}")

if __name__ == "__main__":
    # 请修改为你的实际文件路径
    file1 = r"C:\Users\admin\Desktop\json\12.json"
    file2 = r"C:\Users\admin\Desktop\json\3.json"
    output = r"C:\Users\admin\Desktop\json\123.json"

    merge_annotations_by_name(file1, file2, output)

合并完成（按名称去重）！新文件已保存至：C:\Users\admin\Desktop\json\123.json
